# Lab 1 — Perception Audit and Five Encodings
### SDA-DSC-112 · Module 1 · Perception Science and Visual Encoding · **50 minutes**

| | |
|---|---|
| **Objective** | Encode one Tayseer metric five different ways, rank the encodings by decoding accuracy for a stated task, and run a pre-attentive audit on a deliberately bad dashboard |
| **Dataset** | `data/core/tayseer_services.csv` · `data/dimensions/dim_regions.csv` · `data/instructional/encoding_accuracy_ranking.csv` |
| **Output** | Five encodings, a ranked justification, a written audit, one corrected chart with a lie factor of 1.0 |

| Task | Minutes |
|---|---|
| 1 · Load the golden thread and state the task | 5 |
| 2 · Encode the same series five ways | 15 |
| 3 · Rank your five encodings against Cleveland–McGill | 10 |
| 4 · Perception audit of the bad dashboard | 10 |
| 5 · One corrected chart, exactly one pre-attentive pop | 5 |
| 6 · Lie factor and the benchmark table | 5 |

### Setup

`tayseer_viz.py` sits next to this notebook. It holds the course theme (Okabe–Ito,
colour-blind safe), the correctly-implemented **non-additive measures**, and
`sorted_bar()` — the chart function with the five design principles baked in.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tayseer_viz as tv
tv.apply_theme()

df = tv.load_facts()
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.columns.tolist())

# Your figures go to out/starter/ so they never overwrite the reference set.
tv.set_out_dir(tv.OUT / "starter")

---
## Task 1 — Load the golden thread and state the task *(5 min)*

Load `tayseer_services.csv` and compute **latest-month `digital_adoption_pct` by region**.

State the analytical task in one sentence before you draw anything:

> *"Let a Deputy Minister rank the 13 regions and spot the laggards."*

**The trap you must not fall into.** `digital_adoption_pct` is a **non-additive**
measure recorded at the `month × region × service_category` grain and repeated
across that segment's four channel rows. `AVERAGE()`-ing it gives the wrong number.
Recompute it from `unique_users`, exactly as `kpi_targets.csv` KPI-01 specifies:

```
SUM(digital_adoption_pct / 100 * unique_users) / SUM(unique_users) * 100
```

In [ ]:
latest = df[df.month == df.month.max()]
print("Latest month:", latest.month.iloc[0].date())

# --- TODO -----------------------------------------------------------------
# (a) national_weighted : recompute adoption from unique_users.
#     Hint: tv.adoption(latest)  — or write the SUMPRODUCT yourself, it is two lines.
# (b) national_naive    : the deliberately wrong AVERAGE of the column.
# (c) region_adoption   : one row per region, columns ['region', 'adoption_pct'],
#     sorted highest-first.  Hint: tv.by(frame, "region", "digital_adoption_pct")
national_weighted = None
national_naive = None
region_adoption = None
# --------------------------------------------------------------------------

if region_adoption is None or national_weighted is None:
    print("TODO not implemented yet.")
    print("Expected: national_weighted = 63.8, national_naive = 61.2,")
    print("          region_adoption shape (13, 2) -> ['region', 'adoption_pct']")
else:
    print(f"weighted national adoption : {national_weighted:.1f}%   <- correct")
    print(f"naive AVERAGE of the column: {national_naive:.1f}%   <- the bug")
    assert region_adoption.shape == (13, 2), region_adoption.shape
    assert round(national_weighted, 1) == 63.8, "recompute from unique_users"
    assert round(national_naive, 1) == 61.2
    print("ok")
    display(region_adoption)

---
## Task 2 — Encode the same series five ways *(15 min)*

Same thirteen numbers, five encodings:

| | Encoding | Cleveland–McGill channel | Rank |
|---|---|---|---|
| (a) | Sorted horizontal bar | Position on a common scale | 1 |
| (b) | Unsorted vertical bar | Position on a common scale (unordered) | 1 |
| (c) | Pie chart | Angle / area | 4–5 |
| (d) | Bubble / area | Area | 5 |
| (e) | Single-row heatmap | Colour saturation | 7 |

Build all five and save them. **(c) is the one chart in this course you are allowed to
draw badly — it exists to be critiqued.**

In [ ]:
d = region_adoption.sort_values("adoption_pct") if region_adoption is not None else None

fig, axes = plt.subplots(3, 2, figsize=(13, 13))
axes = axes.ravel()

# (a) DONE FOR YOU — the reference implementation, position on a common scale.
if d is not None:
    tv.sorted_bar(region_adoption, "region", "adoption_pct",
                  highlight=tv.SEVERE,
                  title="(a) Sorted bar — rank is free",
                  xlabel="Digital adoption (%)", ax=axes[0])

# --- TODO -----------------------------------------------------------------
# Fill panels (b)-(e) with the SAME 13 values:
#   axes[1]  (b) unsorted vertical bar  -> axes[1].bar(...);  set_ylim(0, 100)
#   axes[2]  (c) pie                    -> axes[2].pie(...)      [deliberately bad]
#   axes[3]  (d) bubble / area          -> axes[3].scatter(..., s=value**2 * 0.9)
#   axes[4]  (e) single-row heatmap     -> axes[4].imshow(values[None, :], cmap="Blues")
# Give each panel a title naming its Cleveland-McGill channel.
ENCODINGS_DONE = False
# --------------------------------------------------------------------------

if not ENCODINGS_DONE:
    for i, lbl in zip(range(1, 5), ["(b) unsorted bar", "(c) pie",
                                    "(d) bubble/area", "(e) heatmap row"]):
        axes[i].text(.5, .5, f"TODO: {lbl}", ha="center", va="center",
                     fontsize=13, color="#999999")
        axes[i].axis("off")
    print("TODO not implemented yet — expected: 5 filled panels, saved to out/lab1_five_encodings.png")

axes[5].axis("off")
fig.suptitle("Task 2 — one metric, five encodings", x=0.02, ha="left",
             fontsize=16, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
print(tv.save_fig(fig, "lab1_five_encodings"))
plt.show()

---
## Task 3 — Rank your five encodings against Cleveland–McGill *(10 min)*

`encoding_accuracy_ranking.csv` ships the hierarchy with a Tayseer example per channel.
Load it, then rank your five charts **for the stated task** and justify each position by
naming the channel and its rank — not by taste.

In [ ]:
enc = tv.load("encoding_accuracy_ranking.csv")
display(enc[["rank", "channel_en", "decoding_accuracy",
             "relative_accuracy_index", "tayseer_example", "warning"]])

# --- TODO -----------------------------------------------------------------
# Build `my_ranking`: one row per chart (a)-(e), best-first for THIS task.
# Columns: ['my_rank', 'chart', 'channel', 'cm_rank', 'justification'].
# The justification must name the Cleveland-McGill channel, not your preference.
my_ranking = None
# --------------------------------------------------------------------------

if my_ranking is None:
    print("TODO not implemented yet — expected: DataFrame (5, 5), best chart at my_rank 1.")
else:
    assert list(my_ranking.columns) == ["my_rank", "chart", "channel", "cm_rank", "justification"]
    assert len(my_ranking) == 5
    display(my_ranking.sort_values("my_rank"))

---
## Task 4 — Perception audit of the bad dashboard *(10 min)*

The course asset `bad_dashboard_m1.png` is reproduced here **in code**, so the audit
target is reproducible and you can see exactly which sins were committed.

Write the audit: every place a pre-attentive channel is **wasted** on chrome, and every
place the **signal is under-encoded**.

The audit target, built in code so it is reproducible:

In [ ]:
def build_bad_dashboard(region_adoption, path="lab1_bad_dashboard"):
    """Deliberately terrible. Every choice here is a violation we will name."""
    import matplotlib.patches as mpatches
    with plt.rc_context({"axes.grid": True, "axes.spines.top": True,
                         "axes.spines.right": True, "axes.spines.left": True,
                         "axes.titlelocation": "center"}):
        fig, ax = plt.subplots(2, 2, figsize=(12, 8),
                               facecolor="#FFF3C4")          # sin: background fill
        rainbow = plt.get_cmap("hsv")(np.linspace(0, .9, 13))

        # panel 1 — truncated-axis rainbow bar
        a = region_adoption.sort_values("region")
        ax[0, 0].bar(a.region, a.adoption_pct, color=rainbow,
                     edgecolor="black", linewidth=2)          # sin: heavy borders
        ax[0, 0].set_ylim(45, 70)                             # sin: truncated axis
        ax[0, 0].grid(color="#FF00FF", lw=1.4)                # sin: rainbow gridlines
        ax[0, 0].set_title("Digital Adoption by Region")      # sin: topic title
        ax[0, 0].tick_params(axis="x", rotation=90)

        # panel 2 — exploded pie with shadow
        ax[0, 1].pie(a.adoption_pct, labels=a.region, colors=rainbow, shadow=True,
                     explode=[.06] * 13, textprops={"fontsize": 6})
        ax[0, 1].set_title("Adoption Share")                  # sin: false part-to-whole

        # panel 3 — a wall of gauges, faked
        ax[1, 0].axis("off")
        for i, (r, v) in enumerate(zip(a.region[:6], a.adoption_pct[:6])):
            x, y = i % 3, -(i // 3)
            ax[1, 0].add_patch(mpatches.Wedge((x, y), .38, 0, 180,
                                              width=.14, color="#DDDDDD"))
            ax[1, 0].add_patch(mpatches.Wedge((x, y), .38, 180 - v * 1.8, 180,
                                              width=.14, color=rainbow[i]))
            ax[1, 0].text(x, y - .22, f"{r}\n{v:.0f}%", ha="center", fontsize=7)
        ax[1, 0].set_xlim(-.6, 2.6); ax[1, 0].set_ylim(-1.6, .6)
        ax[1, 0].set_title("KPI Gauges")                      # sin: 6 gauges, no hierarchy

        # panel 4 — legend-over-plot spaghetti
        for i, (r, g) in enumerate(a.groupby("region")):
            ax[1, 1].plot([0, 1, 2], [g.adoption_pct.iloc[0] - 6,
                                      g.adoption_pct.iloc[0] - 3,
                                      g.adoption_pct.iloc[0]],
                          color=rainbow[i], lw=2, label=r)
        ax[1, 1].legend(ncol=3, fontsize=6, loc="center")     # sin: legend over the data
        ax[1, 1].set_title("Trend")
        fig.suptitle("TAYSEER PERFORMANCE DASHBOARD", fontsize=20, weight="bold",
                     color="#B00020")
        fig.tight_layout()
    print(tv.save_fig(fig, path))
    return fig

if region_adoption is None:
    print("Complete Task 1 first — the audit target needs `region_adoption`.")
else:
    _ = build_bad_dashboard(region_adoption)
    plt.show()

In [ ]:
# --- TODO -----------------------------------------------------------------
# Write the audit as a DataFrame with columns:
#   ['class', 'finding', 'why_it_fails_perceptually', 'fix']
# `class` is either 'Wasted pre-attentive channel' or 'Signal under-encoded'.
# Aim for at least 7 wasted channels and at least 4 under-encoded signals.
# Name the PERCEPTUAL failure, not the aesthetic one.
audit = None
# --------------------------------------------------------------------------

if audit is None:
    print("TODO not implemented yet — expected: DataFrame with >= 11 rows, 4 columns,")
    print("  class in {'Wasted pre-attentive channel', 'Signal under-encoded'}")
else:
    assert list(audit.columns) == ["class", "finding", "why_it_fails_perceptually", "fix"]
    assert (audit["class"] == "Wasted pre-attentive channel").sum() >= 7, "look again"
    assert (audit["class"] == "Signal under-encoded").sum() >= 4
    display(audit)
    print("ok —", len(audit), "findings")

---
## Task 5 — One corrected chart, exactly one pre-attentive pop *(5 min)*

Rebuild panel 1 honestly. The rules: baseline at zero, sorted by value, grey context,
**exactly one** accent carrying the message, the 65% target as a reference line, and a
title that states the conclusion rather than the subject.

In [ ]:
# --- TODO -----------------------------------------------------------------
# Build ONE corrected chart. Requirements (all five are graded):
#   * horizontal bars, x axis starting at 0
#   * sorted by value
#   * grey context + exactly ONE accent colour, on the laggards
#   * a dashed reference line at tv.TARGET_ADOPTION (65)
#   * a takeaway title (the conclusion, not the topic)
# `tv.sorted_bar(...)` does all five if you pass highlight=, reference=, title=.
CORRECTED_DONE = False
# --------------------------------------------------------------------------

if not CORRECTED_DONE:
    print("TODO not implemented yet — expected: one figure saved to")
    print("  out/lab1_corrected_single_pop.png, with exactly one accent colour.")
else:
    pass  # your fig / tv.save_fig(fig, "lab1_corrected_single_pop") goes above

---
## Task 6 — Lie factor and the benchmark table *(5 min)*

Tufte's **lie factor** = (size of effect shown in the graphic) ÷ (size of effect in the data).

Measure it on the bad dashboard's truncated panel, then confirm your corrected chart
scores **1.0**. Finish by filling the Module 1 benchmark table.

In [ ]:
lo = region_adoption.adoption_pct.min() if region_adoption is not None else None
hi = region_adoption.adoption_pct.max() if region_adoption is not None else None
BASELINE_BAD = 45.0     # where the bad dashboard clipped its axis

# --- TODO -----------------------------------------------------------------
# lie factor = visual ratio / data ratio.
#   data_ratio         = hi / lo
#   visual_ratio_bad   = (hi - BASELINE_BAD) / (lo - BASELINE_BAD)
#   visual_ratio_fixed = (hi - 0) / (lo - 0)
# Then: lie_bad, lie_fixed. Plot the honest/distorted pair side by side and save to
# out/lab1_lie_factor.png. Finally build `benchmark`, the Module 1 table.
lie_bad = None
lie_fixed = None
benchmark = None
# --------------------------------------------------------------------------

if lie_fixed is None:
    print("TODO not implemented yet — expected: lie_bad about 3.4x, lie_fixed exactly 1.00,")
    print("  and a benchmark DataFrame with one row per chart you built.")
else:
    print(f"lie factor, truncated : {lie_bad:.1f}x")
    print(f"lie factor, honest    : {lie_fixed:.2f}x")
    assert abs(lie_fixed - 1.0) < 0.05, "an honest bar chart must score 1.0"
    if benchmark is not None:
        display(benchmark)

---
## Lab 1 checklist

- [ ] Task 1 — both national numbers printed, and you can say *why* they differ
- [ ] Task 2 — five encodings saved to `out/lab1_five_encodings.png`
- [ ] Task 3 — five charts ranked, each justified by its Cleveland–McGill channel
- [ ] Task 4 — audit lists ≥ 7 wasted pre-attentive channels and ≥ 4 under-encoded signals
- [ ] Task 5 — corrected chart: zero baseline, sorted, one accent, target line, takeaway title
- [ ] Task 6 — lie factor of your corrected chart is **1.00**

Save your work as `lab1_solution.ipynb`.